## 1. Initialize the project

Create the working context for the Project, if it has not already been created. The project serves as a placeholder for the code, data, and management of data operations inside the Digitalhub platform.

In [3]:
import digitalhub as dh
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
project = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## 2. Setting up LLM
NeMo guardrails can be used with any LLM that supports the OpenAI-compatible APIs. It is possible to use external LLM API or deploy a custom LLM using the LLM Serving Runtimes. Fetch the "llm" operation in the project. 

In [ ]:
llm_function = project.get_function("llm")

In [ ]:
llm_run = llm_function.run(action="serve")

Check the function is up and running: see the model exposed.

In [ ]:
import requests

BASE_URL = llm_run.refresh().status.service["url"]

res = requests.get(f"{BASE_URL}/models")
res.json()

You should see the list of models exposed by Kube AI including the deployed model.

Test the function is up and running: make a completion call

In [ ]:
MODEL = llm_run.status.openai["model"]
data = {
    "model": MODEL,
    "prompt": "Hello"
  }

res = requests.post(f"{BASE_URL}/completions", json=data)
res.json()

## 3. Setting up the Guardrail
We will use a Guardrail implementation based on GuardrailsAI framework. The framework targets LLM applications and provides a collection of predefined validators that can be used to enforce constraints on the output of the LLM.

Specifically, the function uses Toxic Language validator and blocks the inappropriate requests.

In [ ]:
guardrail_function = project.get_function("toxic-guardrail")

Note that the deployment relies on the environment that defines the details of the LLM. We use OpenAI compatible API so the model engine is set to openai. We also use the MAIN_MODEL_BASE_URL to define the base URL of the LLM API. Finally, we use the OPENAI_API_KEY to define the API key to access the OpenAI API (necessary for the API to work correctly).

## 4. Build the function image

The GuardrailsAI library is based on predefined or custom guardrail validators. Predefined validators may be obtained from the Guardrails AI hub, downloaded and deployed. In this scenario, we will use predefined guardrails that should be integrated in the service. To make it efficient, the guardrails should be integrated in the underlying container image and we will build it for this function.

First, we need an API KEY from Guardrails AI to access the hub. We will add to the project as a secret.

In [ ]:
secret = project.new_secret(name="GUARDRAILS_API_KEY", secret_value="value")

To build the image for the function we will need to add some instructions to use Git, to authenticate to the hub, and to install the specific validator (toxic_language).

In [ ]:
build_run = guardrail_function.run(
    action="build", 
    secrets=["GUARDRAILS_API_KEY"],
    instructions=[
        "/opt/nuclio/uv/uv pip install --system  typer==0.9.0 click==8.1.7 guardrails-ai==0.5.0",
        "apt-get update && apt-get install -y git",
        "--mount=type=secret,id=GUARDRAILS_API_KEY,env=GUARDRAILS_API_KEY guardrails configure --enable-metrics --enable-remote-inferencing --token $GUARDRAILS_API_KEY",
        "guardrails hub install hub://guardrails/toxic_language"
        ]
    )

Check the function is up and running. Then, we need to deploy the guard and protect LLM with it.

In [ ]:
guardrail_run = build_run.refresh().run(action="serve")

## 4. Use the Guardrail to Protect the API

To protect the service instance with guardrails, we rely on the corresponding gateway and use Envoy Gateway extension for the runs. Specifically, when the service enable the extension, we obtain:

- the service exposed also behind the preconfigured service Envoy gateway (see the gatewayInfo in service status);
- if the guardrails are configured, the gateway controls the traffic using the ExtProc extension that interacts with the guardrails to implement pre/post processing logic.

In [ ]:
llm_run = llm_function.run(action="serve", extensions=[{
    "kind": "envoygw",
    "name": "gw",
    "spec": {
        "guardrails": [guardrail_run.refresh().status.service['url']]
    }
}])

The protected endpoint of model gateway may be obtained as follows:

In [ ]:
PROTECTED_ENDPOINT = f"http://{run.status.gatewayInfo['gatewayEndpoint']}/v1"

Test the protected service:

In [ ]:
import requests

MODEL = llm_run.status.openai["model"]
data = {
    "model": MODEL,
    "prompt": "My landlord is an asshole!"
  }

res = requests.post(f"http://{PROTECTED_ENDPOINT}/completions", json=data)
res.text